# Litigi per-user analysis

In [ ]:
from pathlib import Path

from subreddit_lens import load_config

# Locate the analytics directory, which holds subreddit-lens.toml, so paths
# work regardless of the working directory.
ANALYTICS_DIR = next(
    p
    for p in [Path.cwd(), Path.cwd() / "analytics", *Path.cwd().parents]
    if (p / "subreddit-lens.toml").exists()
)
config = load_config(ANALYTICS_DIR / "subreddit-lens.toml")
DATA_DIR = config.data_dir
OUTPUT_DIR = config.output_dir
OUTPUT_DIR.mkdir(exist_ok=True)
# r/litigi is an Italian subreddit: analyse hours in local time.
TZ = config.timezone
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from subreddit_lens import (
    compute_posting_habits_pdf,
    js_similarity,
    load_comments,
    preprocess,
)
from stop_words import get_stop_words

stop = get_stop_words("italian")
from time import time
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA
from scipy.sparse.linalg import eigsh
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import rbf_kernel
from subreddit_lens.constants import DEFAULT_EXCLUDED_AUTHORS
from subreddit_lens.viz import order_by_similarity, plot_similarity_matrix
import plotly.graph_objects as go
import numpy as np
import plotly.express as px

In [ ]:
filename = DATA_DIR / "litigi_comments.parquet"
litigi = load_comments(filename)
litigi['created_dt'] = pd.to_datetime(litigi['created_utc'], unit='s', utc=True).dt.tz_convert(TZ)
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))

litigi['created_hour']=litigi['created_dt'].dt.hour

litigi.head()

# Comments per user

In [ ]:
top_users=litigi.groupby('author').size().sort_values(ascending=False).reset_index().rename(columns={0:'comments'})
top_users=top_users[~top_users['author'].isin(DEFAULT_EXCLUDED_AUTHORS)]
top_users


X = 150
author_list = top_users['author'][:X]
x_grid=np.linspace(0, 24, 1000)
author_densities=compute_posting_habits_pdf(litigi, author_list, x_grid, tz=TZ)

In [ ]:

X = 150
author_list = top_users['author'][:X]
x_grid=np.linspace(0, 24, 1000)
author_densities=compute_posting_habits_pdf(litigi, author_list, x_grid, tz=TZ)
# Authors with a density, in the order of the similarity matrix rows below.
author_list=list(author_densities)

In [ ]:
len(author_densities['SpiegoLeDiscussioni'])

In [ ]:
fig = go.Figure()

for author in author_list:
    density = author_densities[author]
    
    # Add trace to plot
    fig.add_trace(go.Scatter(
        x=x_grid,
        y=density,
        mode='lines',
        name=author,
        line=dict(width=1)
    ))

# Configure plot layout
fig.update_layout(
    title='Activity Distribution with Periodic Boundary Conditions',
    xaxis=dict(
        title='Hour of Day',
        range=[0, 24],
        dtick=2
    ),
    yaxis=dict(title='Density'),
    legend=dict(title='Authors'),
    width=1400,
    height=800,
    template='plotly_white'
)

fig.show()

In [ ]:
# Create a similarity matrix using Jensen-Shannon divergence
n_authors = len(author_list)
similarity_matrix = js_similarity(author_densities)


# Create a DataFrame for the similarity matrix
similarity_df = pd.DataFrame(similarity_matrix, index=author_list, columns=author_list)


# Get permutation order
optimal_order = order_by_similarity(similarity_matrix)

reindexed_author_list=list(similarity_df.columns.to_numpy()[optimal_order])

similarity_df = pd.DataFrame(similarity_matrix[optimal_order][:, optimal_order], index=reindexed_author_list, columns=reindexed_author_list)

fig= plot_similarity_matrix(similarity_df,title="Author posting habits similarity matrix",axis_title="Author")

fig.show()

In [ ]:
threshold=1e-15
k=20
# Normalised affinity D^-1/2 S D^-1/2 (the "pseudo Laplacian" I - L); its top
# eigenvectors give the spectral embedding.
degree = similarity_matrix.sum(axis=1)
d_inv_sqrt = 1 / np.sqrt(degree)
M = d_inv_sqrt[:, None] * similarity_matrix * d_inv_sqrt[None, :]
eivals,eivecs=eigsh(np.where(M < threshold, 0, M),k=k)

In [ ]:
eivec_flipped=np.flip(eivecs,1)
n_components=k
fig,axes=plt.subplots(1,1,squeeze=False,figsize=(20,7),sharex=True)
for i in range(n_components):
    #plt.figure(figsize=(20,5))
    #ax=plt.subplot()
    ax=axes[0][0]
    ax.scatter(range(len(eivec_flipped[:,1])),eivec_flipped[eivec_flipped[:,i].argsort()][:,i],s=4,label=f"{i+1}th eigenvector",alpha=.5)
plt.legend()
llim=0.05
#plt.ylim(top=llim,bottom=-llim)
plt.title("Sorted eigenvectors")
plt.show()

In [ ]:
vec=np.zeros(len(author_list))
k_max=k
fig,axes=plt.subplots(k_max,1,squeeze=False,figsize=(20,15),sharex=True)

for i in range(k_max):
    n_eigenvectors=i+1
    vec=np.sum(normalize(eivecs[:,-n_eigenvectors:],axis=1),axis=1)
    ax=axes[i][0]
    ax.scatter(range(len(author_list)),vec[vec.argsort()],s=2,label=f"Sum, ncomp = {n_eigenvectors}")
    ax.legend()
    ax.grid()

#ax.set_title(f"{i} eigenvector")

#plt.ylim(top=0.2,bottom=-.2)
plt.suptitle("Sorted eigenvectors")
plt.show()

In [ ]:
n=2
vec=np.zeros(len(author_list))
vec=np.sum(normalize(eivecs[:,-n:],axis=1),axis=1)
index_rearranged=vec.argsort()

In [ ]:

grouped_by_user=litigi[litigi['author'].isin(author_list)].groupby('author')


In [ ]:
hour=litigi[litigi['author'].isin(author_list)].groupby(['author','created_hour']).count()
user_comments_binned=hour[['body']].reset_index().pivot_table(index='author',columns='created_hour',aggfunc='sum').reset_index().fillna(0)
user_comments_binned

In [ ]:
user_comments_norm=user_comments_binned.set_index('author').droplevel(0,axis=1).div(user_comments_binned.set_index('author').droplevel(0,axis=1).max(axis=1),axis=0)
user_comments_norm

In [ ]:
user_comments=grouped_by_user['body_preprocessed'].agg(lambda x: ' '.join(x)).reset_index()
users=list(user_comments['author'])

# NLP analysis

In [ ]:
user_comments

In [ ]:
t0 = time()
vectorizer = TfidfVectorizer(stop_words=stop,max_features=1000)
X_tfidf = vectorizer.fit_transform(user_comments['body_preprocessed'])
print(f"vectorization done in {time() - t0:.3f} s")
print(f"n_samples: {X_tfidf.shape[0]}, n_features: {X_tfidf.shape[1]}")

In [ ]:
from minisom import MiniSom

combined_features=np.concatenate((Normalizer().fit_transform(X_tfidf.toarray(),'l1'), user_comments_norm.loc[users].values), axis=1)

som_shape=(30,30)

som = MiniSom(som_shape[0], som_shape[1], combined_features.shape[1], sigma=1.5, learning_rate=.1, activation_distance='euclidean',
              topology='rectangular', neighborhood_function='gaussian', random_seed=10)

som.train_batch(combined_features, 1000, verbose=True)

In [ ]:
plt.pcolor(som.distance_map().T, cmap='bone_r')  # plotting the distance map as background

In [ ]:
labels=user_comments['author']

In [ ]:



lsa = make_pipeline(TruncatedSVD(n_components=100), Normalizer(copy=False))
t0 = time()
X_lsa = lsa.fit_transform(X_tfidf)
explained_variance = lsa[0].explained_variance_ratio_.sum()

print(f"LSA done in {time() - t0:.3f} s")
print(f"Explained variance of the SVD step: {explained_variance * 100:.1f}%")

In [ ]:
# RBF (Gaussian) similarity between users in the LSA space.
Sm = rbf_kernel(X_lsa, gamma=1.0)
Sm

In [ ]:
# Plot the similarity matrix as a heatmap


similarity_df = pd.DataFrame(Sm, index=users, columns=users)

fig = go.Figure(data=go.Heatmap(
    z=similarity_df.values,
    x=similarity_df.columns,
    y=similarity_df.index,
    colorscale='Viridis',
    zmax=1,
    colorbar=dict(title='Similarity')
))

fig.update_layout(
    title='Author Posting Habit Similarity Matrix - Lexical',
    xaxis=dict(title='Authors'),
    yaxis=dict(title='Authors'),
    width=1200,
    height=1200,
    template='plotly_white'
)

fig.show()

In [ ]:
fig = plot_similarity_matrix(similarity_df,title='Author Posting Habit Similarity Matrix - Lexical',axis_title="Author",order=True)

fig.show()

In [ ]:
result = list(similarity_df.stack()[similarity_df.stack() > .70].index)
tpl=[a for a in result if (a[0] != a[1])]
unique_tuples = set()
filtered_tuples = []

for t in tpl:
    if (t[1], t[0]) not in unique_tuples:
        unique_tuples.add(t)
        filtered_tuples.append(t)
filtered_tuples

# Combining results

In [ ]:
tfidf_similarity = pd.DataFrame(Sm, index=users, columns=users)
# Rows of similarity_matrix follow author_list, not users (sorted by name).
posting_time_similarity = pd.DataFrame(similarity_matrix, index=author_list, columns=author_list)
tfidf_similarity_aligned = tfidf_similarity.reindex_like(posting_time_similarity).fillna(0)

In [ ]:
alpha=.2
full_matrix=alpha*posting_time_similarity.values+(1-alpha)*tfidf_similarity_aligned.values
full_matrix=full_matrix/full_matrix.max()
full_matrix

In [ ]:
# Plot the similarity matrix as a heatmap


similarity_df =pd.DataFrame(full_matrix,index=posting_time_similarity.index,columns=posting_time_similarity.columns)

fig = go.Figure(data=go.Heatmap(
    z=similarity_df.values,
    x=similarity_df.columns,
    y=similarity_df.index,
    colorscale='Viridis',
    zmax=1,
    colorbar=dict(title='Similarity')
))

fig.update_layout(
    title='Author Posting Habit Similarity Matrix - Combined',
    xaxis=dict(title='Authors'),
    yaxis=dict(title='Authors'),
    width=1200,
    height=1200,
    template='plotly_white'
)

fig.show()

In [ ]:
fig=plot_similarity_matrix(similarity_df,title='Similarity - Combined',order=True)
fig.show()

In [ ]:
for col in sorted(similarity_df.columns):
    print(f"Il più simile a {col} è: {similarity_df[col].sort_values(ascending=False).index[1]}")

In [ ]:
result = list(similarity_df.stack()[similarity_df.stack() > .87].index)
[a for a in result if (a[0] != a[1])]

In [ ]:

tfidf_similarity_aligned

In [ ]:
pca=make_pipeline(lsa,PCA(n_components=2))
X_pca = pca.fit_transform(X_tfidf)
X_pca

In [ ]:
px.scatter(x=X_pca[:,0],y=X_pca[:,1],labels=labels,color=labels)

In [ ]:
user_comments

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np
# Vectorize text
vectorizer = CountVectorizer(stop_words=stop)
X = vectorizer.fit_transform(litigi[:10000]['body_preprocessed'])

# Apply LDA
lda = LatentDirichletAllocation(n_components=10, random_state=42)
lda.fit(X)

# Display topics
topics = np.argsort(lda.components_, axis=1)[:, -5:]
print("Top words per topic:")
for i, topic in enumerate(topics):
    print(f"Topic {i}: {[vectorizer.get_feature_names_out()[j] for j in topic]}")